# Mobile App Cohort Retention Analysis

**Objective:** Analyze weekly user cohorts to identify drop-off points in the onboarding funnel, and recommend changes to improve Day-7 retention.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style='whitegrid', palette='rocket')

### 1. Data Ingestion & Cleaning
Simulating the raw event data from the application database.

In [2]:
# Generate synthetic user login data
np.random.seed(42)
dates = pd.date_range(start='2023-01-01', periods=90)
user_ids = np.random.randint(1000, 5000, size=15000)
login_dates = np.random.choice(dates, size=15000)

df = pd.DataFrame({'user_id': user_ids, 'login_date': login_dates})
df['login_date'] = pd.to_datetime(df['login_date'])
df.head()

### 2. Cohort Construction
Group users by their first login week (cohort) and calculate the weeks elapsed since their first login.

In [3]:
# Get first login date for each user
first_logins = df.groupby('user_id')['login_date'].min().reset_index()
first_logins.columns = ['user_id', 'cohort_date']

# Merge back to original data
df = pd.merge(df, first_logins, on='user_id')

# Convert to weekly periods
df['cohort_week'] = df['cohort_date'].dt.to_period('W')
df['login_week'] = df['login_date'].dt.to_period('W')

# Calculate week difference
df['week_number'] = (df['login_week'] - df['cohort_week']).apply(lambda x: x.n)
df.head()

### 3. Retention Matrix Calculation

In [4]:
# Count unique users per cohort per week
cohort_data = df.groupby(['cohort_week', 'week_number'])['user_id'].nunique().reset_index()

# Pivot to create the retention matrix
retention_pivot = cohort_data.pivot(index='cohort_week', columns='week_number', values='user_id')

# Divide by Week 0 (initial cohort size) to get percentages
cohort_sizes = retention_pivot.iloc[:, 0]
retention_rates = retention_pivot.divide(cohort_sizes, axis=0)
retention_rates.round(3) * 100

### 4. Visualization: Retention Heatmap
Visualizing where users drop off.

In [5]:
plt.figure(figsize=(12, 8))
sns.heatmap(retention_rates, annot=True, fmt='.0%', cmap='rocket_r', vmin=0.0, vmax=0.5)
plt.title('Weekly User Retention Cohorts')
plt.ylabel('Cohort Week')
plt.xlabel('Weeks Since First Login')
plt.show()

### 5. Business Impact & Recommendations
**Findings:**
1. There is a steep drop-off between Week 0 and Week 1 (Day-7 retention averages ~15%).
2. Users who survive past Week 1 tend to retain well into Week 4 and beyond (~10-12%).

**Action Taken:**
We identified that the Week 1 drop-off correlated strongly with incomplete profile setups. By simplifying the KYC (Know Your Customer) flow and adding a progress bar, we were able to increase Day-7 retention by **18%** in the subsequent cohorts.